In [1]:
import cv2
import numpy as np
import pandas as pd
import time

video_path = r"C:\Users\skybl\Desktop\已完成學業\大三上_逢甲\進階專題\水鹿影片\水鹿1.mp4"

# Create VideoCapture object
cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    print("無法打開視頻檔案")
    exit()

# Initialize list to store activity counts and status
activity_data = {"Activity_Count": [], "Status": []}

# Initialize average frame
ret, frame = cap.read()
avg = cv2.blur(frame, (4, 4))
avg_float = np.float32(avg)

start_time = time.time()

while(cap.isOpened()):
    ret, frame = cap.read()

    # If reached end of video
    if ret == False:
        break

    # Blur the frame
    blur = cv2.blur(frame, (4, 4))

    # Calculate difference between current frame and average frame
    diff = cv2.absdiff(avg, blur)

    # Convert image to grayscale
    gray = cv2.cvtColor(diff, cv2.COLOR_BGR2GRAY)

    # Threshold the grayscale image
    ret, thresh = cv2.threshold(gray, 25, 255, cv2.THRESH_BINARY)

    # Use morphology transformations to remove noise
    kernel = np.ones((5, 5), np.uint8)
    thresh = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel, iterations=2)
    thresh = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel, iterations=2)

    # Count non-zero pixels in the thresholded image
    activity_count = np.count_nonzero(thresh)

    # Append activity count and status to data
    activity_data["Activity_Count"].append(activity_count)
    activity_data["Status"].append("Normal")

end_time = time.time()

# Release VideoCapture object
cap.release()
cv2.destroyAllWindows()

print(f"程式運行時間：{end_time - start_time} 秒")

# Convert activity_data to DataFrame
df = pd.DataFrame(activity_data)
print(df)


程式運行時間：4.484112024307251 秒
     Activity_Count  Status
0                 0  Normal
1                 0  Normal
2                 0  Normal
3                 0  Normal
4                 0  Normal
..              ...     ...
594           86118  Normal
595           85717  Normal
596           86033  Normal
597           85927  Normal
598           85877  Normal

[599 rows x 2 columns]


In [7]:
import cv2
import numpy as np
import pandas as pd
import time
import os

# 定義各狀態的資料夾路徑
status_folders = {
    "Normal": r"C:\Users\skybl\Desktop\已完成學業\大三上_逢甲\進階專題\水鹿影片\水路剪輯\N",
    "Aggressive": r"C:\Users\skybl\Desktop\已完成學業\大三上_逢甲\進階專題\水鹿影片\水路剪輯\A",
    "Passive": r"C:\Users\skybl\Desktop\已完成學業\大三上_逢甲\進階專題\水鹿影片\水路剪輯\P",
    # 添加更多狀態和對應的資料夾路徑...
}

# 初始化 DataFrame 來存儲所有影片的數據
all_activity_data = {"Activity_Count": [], "Status": []}

# 遍歷各狀態的資料夾
for status, folder_path in status_folders.items():
    print(f"處理狀態：{status}，資料夾：{folder_path}")

    # 獲取資料夾中所有影片的路徑
    video_paths = [os.path.join(folder_path, file) for file in os.listdir(folder_path) if file.endswith(".mp4")]

    # 遍歷影片路徑列表
    for video_path in video_paths:
        print(f"處理影片：{video_path}")

        # 創建 VideoCapture 物件
        cap = cv2.VideoCapture(video_path)

        if not cap.isOpened():
            print(f"無法打開視頻檔案：{video_path}")
            continue

        # 初始化平均影像
        ret, frame = cap.read()
        avg = cv2.blur(frame, (4, 4))
        avg_float = np.float32(avg)

        start_time = time.time()

        # 初始化活動數據字典
        activity_data = {"Activity_Count": [], "Status": []}

        while(cap.isOpened()):
            ret, frame = cap.read()

            # 若讀取至影片結尾
            if ret == False:
                break

            # 模糊處理
            blur = cv2.blur(frame, (4, 4))

            # 計算目前影格與平均影像的差異值
            diff = cv2.absdiff(avg, blur)

            # 將圖片轉為灰階
            gray = cv2.cvtColor(diff, cv2.COLOR_BGR2GRAY)

            # 篩選出變動程度大於門檻值的區域
            ret, thresh = cv2.threshold(gray, 25, 255, cv2.THRESH_BINARY)

            # 使用型態轉換函數去除雜訊
            kernel = np.ones((5, 5), np.uint8)
            thresh = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel, iterations=2)
            thresh = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel, iterations=2)
            activity_count = np.count_nonzero(thresh)

            # 添加數據到活動數據字典
            activity_data["Activity_Count"].append(activity_count)
            activity_data["Status"].append(status)

        end_time = time.time()

        # 釋放 VideoCapture 物件
        cap.release()

        # 轉換活動數據字典到 DataFrame
        df_video = pd.DataFrame(activity_data)

        # 將當前影片的數據添加到總數據 DataFrame 中
        all_activity_data["Activity_Count"].extend(df_video["Activity_Count"])
        all_activity_data["Status"].extend(df_video["Status"])

        print(f"影片處理時間：{end_time - start_time} 秒")

# 轉換總數據字典到 DataFrame
df_all = pd.DataFrame(all_activity_data)

# 儲存 DataFrame 到 Excel 檔案
excel_filename = "activity_data_all.xlsx"
df_all.to_excel(excel_filename, index=False)
print(f"所有影片的數據已儲存至 {excel_filename}")


處理狀態：Normal，資料夾：C:\Users\skybl\Desktop\已完成學業\大三上_逢甲\進階專題\水鹿影片\水路剪輯\N
處理影片：C:\Users\skybl\Desktop\已完成學業\大三上_逢甲\進階專題\水鹿影片\水路剪輯\N\亂走亂舔.mp4
影片處理時間：6.55955958366394 秒
處理影片：C:\Users\skybl\Desktop\已完成學業\大三上_逢甲\進階專題\水鹿影片\水路剪輯\N\互舔.mp4
影片處理時間：9.386475801467896 秒
處理影片：C:\Users\skybl\Desktop\已完成學業\大三上_逢甲\進階專題\水鹿影片\水路剪輯\N\互舔＋到處晃晃.mp4
影片處理時間：14.757673978805542 秒
處理影片：C:\Users\skybl\Desktop\已完成學業\大三上_逢甲\進階專題\水鹿影片\水路剪輯\N\喝水＋聞屁.mp4
影片處理時間：14.697066068649292 秒
處理影片：C:\Users\skybl\Desktop\已完成學業\大三上_逢甲\進階專題\水鹿影片\水路剪輯\N\抬腳抓癢(無裁切版）.mp4
影片處理時間：5.3749096393585205 秒
處理影片：C:\Users\skybl\Desktop\已完成學業\大三上_逢甲\進階專題\水鹿影片\水路剪輯\N\搖尾巴＋吃草.mp4
影片處理時間：4.0006983280181885 秒
處理影片：C:\Users\skybl\Desktop\已完成學業\大三上_逢甲\進階專題\水鹿影片\水路剪輯\N\正常活動.mp4
影片處理時間：14.616203308105469 秒
處理影片：C:\Users\skybl\Desktop\已完成學業\大三上_逢甲\進階專題\水鹿影片\水路剪輯\N\玩＋聞屁＋舔腳.mp4
影片處理時間：9.872086524963379 秒
處理影片：C:\Users\skybl\Desktop\已完成學業\大三上_逢甲\進階專題\水鹿影片\水路剪輯\N\舔自己.mp4
影片處理時間：7.413878917694092 秒
處理影片：C:\Users\skybl\Desktop\已完成學業\大三上_逢甲\進階專題\水鹿影片\水路剪輯\N\趴下動作.mp4
影片

In [11]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

# 讀取資料
df = pd.read_excel("activity_data_all.xlsx")

# 將狀態轉換為數字形式
df["Status"] = df["Status"].map({"Normal": 1, "Aggressive": 2, "Passive": 0})

# 拆分特徵和標籤
X = df[["Activity_Count"]].values
y = df["Status"].values

# 標準化特徵
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 拆分訓練集和測試集
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# 建立 LSTM 模型
model = Sequential([
    LSTM(units=50, input_shape=(X_train.shape[1], 1)),
    Dense(units=3, activation="softmax")
])

# 編譯模型
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])

# 訓練模型
model.fit(X_train.reshape((X_train.shape[0], X_train.shape[1], 1)), y_train, epochs=10, batch_size=16, validation_split=0.2)

# 評估模型
loss, accuracy = model.evaluate(X_test.reshape((X_test.shape[0], X_test.shape[1], 1)), y_test)
print(f"測試集準確率：{accuracy}")


Epoch 1/10
313/313 [==============================] - 2s 3ms/step - loss: 1.0223 - accuracy: 0.4909 - val_loss: 0.9994 - val_accuracy: 0.4800
Epoch 2/10
313/313 [==============================] - 0s 1ms/step - loss: 0.9936 - accuracy: 0.4964 - val_loss: 0.9964 - val_accuracy: 0.4932
Epoch 3/10
313/313 [==============================] - 0s 1ms/step - loss: 0.9904 - accuracy: 0.5028 - val_loss: 0.9927 - val_accuracy: 0.4940
Epoch 4/10
313/313 [==============================] - 0s 1ms/step - loss: 0.9871 - accuracy: 0.5041 - val_loss: 0.9896 - val_accuracy: 0.4956
Epoch 5/10
313/313 [==============================] - 0s 1ms/step - loss: 0.9823 - accuracy: 0.5074 - val_loss: 0.9829 - val_accuracy: 0.4992
Epoch 6/10
313/313 [==============================] - 0s 1ms/step - loss: 0.9758 - accuracy: 0.5145 - val_loss: 0.9765 - val_accuracy: 0.5048
Epoch 7/10
313/313 [==============================] - 0s 2ms/step - loss: 0.9681 - accuracy: 0.5181 - val_loss: 0.9683 - val_accuracy: 0.5080
Epoch 

In [2]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
import pandas as pd
import numpy as np

# 讀取資料
df = pd.read_excel("activity_data_all.xlsx")

# 將狀態轉換為數字形式
df["Status"] = df["Status"].map({"Normal": 1, "Aggressive": 2, "Passive": 0})

# 拆分特徵和標籤
X = df[["Activity_Count"]].values
y = df["Status"].values

# 標準化特徵
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 拆分訓練集和測試集
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# 建立 LSTM 模型
model = Sequential([
    LSTM(units=50, input_shape=(X_train.shape[1], 1), return_sequences=True),
    Dropout(0.2),
    LSTM(units=50),
    Dropout(0.2),
    Dense(units=3, activation="softmax")
])

# 編譯模型
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])

# 訓練模型
model.fit(X_train.reshape((X_train.shape[0], X_train.shape[1], 1)), y_train, epochs=50, batch_size=16, validation_split=0.2)

# 用模型預測測試集
y_pred = model.predict(X_test.reshape((X_test.shape[0], X_test.shape[1], 1)))
# 將預測結果轉換為類別標籤
y_pred_classes = np.argmax(y_pred, axis=1)

# 計算 RMSE
rmse = np.sqrt(mean_squared_error(y_test, y_pred_classes))
print(f"RMSE：{rmse}")

from sklearn.metrics import f1_score

# 计算 F1 分数
f1 = f1_score(y_test, y_pred_classes, average='weighted')
print(f"F1 Score：{f1}")


Epoch 1/50
626/626 [==============================] - 4s 3ms/step - loss: 1.0013 - accuracy: 0.4948 - val_loss: 0.9912 - val_accuracy: 0.4944
Epoch 2/50
626/626 [==============================] - 1s 2ms/step - loss: 0.9802 - accuracy: 0.5157 - val_loss: 0.9743 - val_accuracy: 0.5136
Epoch 3/50
626/626 [==============================] - 1s 2ms/step - loss: 0.9564 - accuracy: 0.5258 - val_loss: 0.9535 - val_accuracy: 0.5168
Epoch 4/50
626/626 [==============================] - 1s 2ms/step - loss: 0.9454 - accuracy: 0.5245 - val_loss: 0.9390 - val_accuracy: 0.5232
Epoch 5/50
626/626 [==============================] - 1s 2ms/step - loss: 0.9366 - accuracy: 0.5257 - val_loss: 0.9292 - val_accuracy: 0.5148
Epoch 6/50
626/626 [==============================] - 1s 2ms/step - loss: 0.9296 - accuracy: 0.5232 - val_loss: 0.9226 - val_accuracy: 0.5172
Epoch 7/50
626/626 [==============================] - 1s 2ms/step - loss: 0.9257 - accuracy: 0.5191 - val_loss: 0.9269 - val_accuracy: 0.5192
Epoch 

## 如果每秒只擷取一次

In [17]:
import cv2
import numpy as np
import pandas as pd
import time
import os

# 定義各狀態的資料夾路徑
status_folders = {
    "Normal": r"C:\Users\skybl\Desktop\已完成學業\大三上_逢甲\進階專題\水鹿影片\水路剪輯\N",
    "Aggressive": r"C:\Users\skybl\Desktop\已完成學業\大三上_逢甲\進階專題\水鹿影片\水路剪輯\A",
    "Passive": r"C:\Users\skybl\Desktop\已完成學業\大三上_逢甲\進階專題\水鹿影片\水路剪輯\P",
    # 添加更多狀態和對應的資料夾路徑...
}

# 初始化 DataFrame 來存儲所有影片的數據
all_activity_data = {"Activity_Count": [], "Status": []}

# 遍歷各狀態的資料夾
for status, folder_path in status_folders.items():
    print(f"處理狀態：{status}，資料夾：{folder_path}")

    # 獲取資料夾中所有影片的路徑
    video_paths = [os.path.join(folder_path, file) for file in os.listdir(folder_path) if file.endswith(".mp4")]

    # 遍歷影片路徑列表
    for video_path in video_paths:
        print(f"處理影片：{video_path}")

        # 創建 VideoCapture 物件
        cap = cv2.VideoCapture(video_path)

        if not cap.isOpened():
            print(f"無法打開視頻檔案：{video_path}")
            continue

        # 初始化活動數據字典
        activity_data = {"Activity_Count": [], "Status": []}
        
        frame_count = 0  # 計算影格數量
        activity_sum = 0  # 計算每秒活動量總和
        start_time = time.time()  # 開始時間

        while(cap.isOpened()):
            ret, frame = cap.read()

            # 若讀取至影片結尾
            if not ret:
                break
            
            # 計算活動量的程式碼
            # 這裡示範每 25 個影格計算一次活動量，假設影片每秒有 25 幀 (fps=25)
            if frame_count % 30 == 0 and frame_count != 0:
                avg_frame_count = frame_count / 30  # 平均每秒的影格數
                activity_avg = activity_sum / avg_frame_count  # 平均每秒的活動量
                activity_data["Activity_Count"].append(activity_avg)
                activity_data["Status"].append(status)
                activity_sum = 0  # 重設每秒活動量總和
            
            # 模糊處理
            blur = cv2.blur(frame, (4, 4))

            # 計算目前影格與平均影像的差異值
            diff = cv2.absdiff(avg, blur)

            # 將圖片轉為灰階
            gray = cv2.cvtColor(diff, cv2.COLOR_BGR2GRAY)

            # 篩選出變動程度大於門檻值的區域
            ret, thresh = cv2.threshold(gray, 25, 255, cv2.THRESH_BINARY)

            # 使用型態轉換函數去除雜訊
            kernel = np.ones((5, 5), np.uint8)
            thresh = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel, iterations=2)
            thresh = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel, iterations=2)
            activity_count = np.count_nonzero(thresh)

            activity_sum += activity_count  # 累加每秒活動量
            frame_count += 1  # 更新影格數量

        end_time = time.time()

        # 釋放 VideoCapture 物件
        cap.release()

        # 轉換活動數據字典到 DataFrame
        df_video = pd.DataFrame(activity_data)

        # 將當前影片的數據添加到總數據 DataFrame 中
        all_activity_data["Activity_Count"].extend(df_video["Activity_Count"])
        all_activity_data["Status"].extend(df_video["Status"])

        print(f"影片處理時間：{end_time - start_time} 秒")

# 轉換總數據字典到 DataFrame
df_all = pd.DataFrame(all_activity_data)

# 儲存 DataFrame 到 Excel 檔案
excel_filename = "activity_data_all(2).xlsx"
df_all.to_excel(excel_filename, index=False)
print(f"所有影片的數據已儲存至 {excel_filename}")


處理狀態：Normal，資料夾：C:\Users\skybl\Desktop\已完成學業\大三上_逢甲\進階專題\水鹿影片\水路剪輯\N
處理影片：C:\Users\skybl\Desktop\已完成學業\大三上_逢甲\進階專題\水鹿影片\水路剪輯\N\亂走亂舔.mp4
影片處理時間：6.164363145828247 秒
處理影片：C:\Users\skybl\Desktop\已完成學業\大三上_逢甲\進階專題\水鹿影片\水路剪輯\N\互舔.mp4
影片處理時間：9.020064353942871 秒
處理影片：C:\Users\skybl\Desktop\已完成學業\大三上_逢甲\進階專題\水鹿影片\水路剪輯\N\互舔＋到處晃晃.mp4
影片處理時間：16.33426547050476 秒
處理影片：C:\Users\skybl\Desktop\已完成學業\大三上_逢甲\進階專題\水鹿影片\水路剪輯\N\喝水＋聞屁.mp4
影片處理時間：16.40387773513794 秒
處理影片：C:\Users\skybl\Desktop\已完成學業\大三上_逢甲\進階專題\水鹿影片\水路剪輯\N\抬腳抓癢(無裁切版）.mp4
影片處理時間：6.255291700363159 秒
處理影片：C:\Users\skybl\Desktop\已完成學業\大三上_逢甲\進階專題\水鹿影片\水路剪輯\N\搖尾巴＋吃草.mp4
影片處理時間：4.673285961151123 秒
處理影片：C:\Users\skybl\Desktop\已完成學業\大三上_逢甲\進階專題\水鹿影片\水路剪輯\N\正常活動.mp4
影片處理時間：17.081799268722534 秒
處理影片：C:\Users\skybl\Desktop\已完成學業\大三上_逢甲\進階專題\水鹿影片\水路剪輯\N\玩＋聞屁＋舔腳.mp4
影片處理時間：11.63495421409607 秒
處理影片：C:\Users\skybl\Desktop\已完成學業\大三上_逢甲\進階專題\水鹿影片\水路剪輯\N\舔自己.mp4
影片處理時間：8.522236824035645 秒
處理影片：C:\Users\skybl\Desktop\已完成學業\大三上_逢甲\進階專題\水鹿影片\水路剪輯\N\趴下動作.mp4
影片處理時

In [18]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd

# 讀取資料
df = pd.read_excel("activity_data_all(2).xlsx")

# 將狀態轉換為數字形式
df["Status"] = df["Status"].map({"Normal": 1, "Aggressive": 2, "Passive": 0})

# 拆分特徵和標籤
X = df[["Activity_Count"]].values
y = df["Status"].values

# 標準化特徵
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 拆分訓練集和測試集
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# 建立 LSTM 模型
model = Sequential([
    LSTM(units=50, input_shape=(X_train.shape[1], 1), return_sequences=True),
    Dropout(0.2),
    LSTM(units=50),
    Dropout(0.2),
    Dense(units=3, activation="softmax")
])

# 編譯模型
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])

# 訓練模型
model.fit(X_train.reshape((X_train.shape[0], X_train.shape[1], 1)), y_train, epochs=50, batch_size=16, validation_split=0.2)

# 評估模型
loss, accuracy = model.evaluate(X_test.reshape((X_test.shape[0], X_test.shape[1], 1)), y_test)
print(f"測試集準確率：{accuracy}")
# 心得 batch_size 16與8 準確率相差不大，從32減為16 有助於提升準確度

Epoch 1/50
21/21 [==============================] - 4s 38ms/step - loss: 1.0980 - accuracy: 0.3508 - val_loss: 1.0936 - val_accuracy: 0.4512
Epoch 2/50
21/21 [==============================] - 0s 4ms/step - loss: 1.0953 - accuracy: 0.3846 - val_loss: 1.0888 - val_accuracy: 0.4512
Epoch 3/50
21/21 [==============================] - 0s 4ms/step - loss: 1.0929 - accuracy: 0.3785 - val_loss: 1.0841 - val_accuracy: 0.4512
Epoch 4/50
21/21 [==============================] - 0s 4ms/step - loss: 1.0911 - accuracy: 0.3969 - val_loss: 1.0788 - val_accuracy: 0.4024
Epoch 5/50
21/21 [==============================] - 0s 4ms/step - loss: 1.0894 - accuracy: 0.3815 - val_loss: 1.0755 - val_accuracy: 0.4146
Epoch 6/50
21/21 [==============================] - 0s 4ms/step - loss: 1.0881 - accuracy: 0.4062 - val_loss: 1.0703 - val_accuracy: 0.3780
Epoch 7/50
21/21 [==============================] - 0s 4ms/step - loss: 1.0868 - accuracy: 0.3938 - val_loss: 1.0662 - val_accuracy: 0.3902
Epoch 8/50
21/21 [=

## 一個影片一個總活動量

In [6]:
import cv2
import numpy as np
import pandas as pd
import time
import os

# 定義各狀態的資料夾路徑
status_folders = {
    "Normal": r"C:\Users\skybl\Desktop\已完成學業\大三上_逢甲\進階專題\水鹿影片\水路剪輯\N",
    "Aggressive": r"C:\Users\skybl\Desktop\已完成學業\大三上_逢甲\進階專題\水鹿影片\水路剪輯\A",
    "Passive": r"C:\Users\skybl\Desktop\已完成學業\大三上_逢甲\進階專題\水鹿影片\水路剪輯\P",
}

# 初始化 DataFrame 來存儲所有影片的總活動量數據
all_activity_data = {"Total_Activity": [], "Status": [], "Video_Path": []}

# 遍歷各狀態的資料夾
for status, folder_path in status_folders.items():
    print(f"處理狀態：{status}，資料夾：{folder_path}")

    # 遍歷資料夾中的影片
    for file in os.listdir(folder_path):
        if file.endswith(".mp4"):
            video_path = os.path.join(folder_path, file)

            # 創建 VideoCapture 物件
            cap = cv2.VideoCapture(video_path)

            if not cap.isOpened():
                print(f"無法打開視頻檔案：{video_path}")
                continue

            # 初始化活動量總和
            total_nonzero = 0

            # 初始化平均影像
            ret, frame = cap.read()
            avg = cv2.blur(frame, (4, 4))
            avg_float = np.float32(avg)

            while(cap.isOpened()):
                ret, frame = cap.read()
                
                # 若讀取至影片結尾
                if ret == False:
                    break
                
                # 模糊處理
                blur = cv2.blur(frame, (4, 4))

                # 計算目前影格與平均影像的差異值
                diff = cv2.absdiff(avg, blur)

                # 將圖片轉為灰階
                gray = cv2.cvtColor(diff, cv2.COLOR_BGR2GRAY)

                # 篩選出變動程度大於門檻值的區域
                ret, thresh = cv2.threshold(gray, 25, 255, cv2.THRESH_BINARY)

                # 使用型態轉換函數去除雜訊
                kernel = np.ones((5, 5), np.uint8)
                thresh = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel, iterations=2)
                thresh = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel, iterations=2)
                total_nonzero += np.count_nonzero(thresh)

            # 釋放 VideoCapture 物件
            cap.release()

            # 將數據添加到 DataFrame 中
            all_activity_data["Total_Activity"].append(total_nonzero)
            all_activity_data["Status"].append(status)
            all_activity_data["Video_Path"].append(video_path)

# 轉換數據字典到 DataFrame
df_all = pd.DataFrame(all_activity_data)

# 儲存 DataFrame 到 Excel 檔案
excel_filename = "total_activity_by_video.xlsx"
df_all.to_excel(excel_filename, index=False)
print(f"各狀態影片的總活動量數據已儲存至 {excel_filename}")


處理狀態：Normal，資料夾：C:\Users\skybl\Desktop\已完成學業\大三上_逢甲\進階專題\水鹿影片\水路剪輯\N
處理狀態：Aggressive，資料夾：C:\Users\skybl\Desktop\已完成學業\大三上_逢甲\進階專題\水鹿影片\水路剪輯\A
處理狀態：Passive，資料夾：C:\Users\skybl\Desktop\已完成學業\大三上_逢甲\進階專題\水鹿影片\水路剪輯\P
各狀態影片的總活動量數據已儲存至 total_activity_by_video.xlsx
